# Text Readability Assessment Demo

This notebook demonstrates the **Standard Readability Datasets** artifact, which provides standardized text readability data with ground-truth grade-level labels (1-12) from educators.

## What This Artifact Does

- Collects and standardizes 3 high-quality readability datasets (OneStopEnglish, CommonLit, CEFR-SP)
- Provides 12,469 total examples with ground-truth labels from educators (not algorithm-derived)
- Formats data as `{input: text, output: grade_level}` with train/val/test splits
- Includes full, mini, and preview JSON variants for efficient development

## Demo Overview

In this demo, we will:
1. Load a curated subset of readability data
2. Explore the data structure and statistics
3. Visualize the distribution of grade levels
4. Demonstrate a simple readability scoring method (Flesch-Kincaid)
5. Compare the simple method against ground-truth labels

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

print('Dependencies installed successfully!')

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import mean_absolute_error, accuracy_score

print('Imports completed!')

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-703cc9-network-percolation-features-for-text-re/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f'GitHub load failed: {e}')
    
    if os.path.exists('mini_demo_data.json'):
        with open('mini_demo_data.json') as f:
            return json.load(f)
    
    raise FileNotFoundError('Could not load mini_demo_data.json')

print('Data loading helper defined!')

In [ ]:
# Load the demo data
data = load_data()

print('Data loaded successfully!')
print(f'Number of datasets: {len(data["datasets"])}')
print(f'Total examples: {data["metadata"]["total_examples"]}')
print(f'Grade range: {data["metadata"]["grade_range"]}')

## Data Structure Exploration

Let's examine the structure of our readability dataset and look at some example texts.

In [ ]:
# Extract all examples into a flat list
all_examples = []
for dataset in data['datasets']:
    for example in dataset['examples']:
        all_examples.append({
            'text': example['input'],
            'grade_level': int(example['output']),
            'source': example['metadata_source'],
            'id': example['metadata_id']
        })

print(f'Total examples extracted: {len(all_examples)}')
print('\nFirst 3 examples:')
for i, ex in enumerate(all_examples[:3]):
    print(f"\nExample {i+1}:")
    print(f"  Text: {ex['text'][:100]}...")
    print(f"  Grade Level: {ex['grade_level']}")
    print(f"  Source: {ex['source']}")

## Configuration

Set tunable parameters for the readability analysis. For this demo, we use minimal values.

In [ ]:
# Configuration parameters
CONFIG = {
    'min_grade': 1,          # Minimum grade level to include
    'max_grade': 12,         # Maximum grade level to include
    'sample_size': 10,       # Number of examples to use (mini dataset)
    'random_seed': 42        # For reproducibility
}

print('Configuration:')
for key, value in CONFIG.items():
    print(f'  {key}: {value}')

## Simple Readability Scoring Method

We'll implement a simple **Flesch-Kincaid Grade Level** formula as a baseline readability scorer:

```
FKGL = 0.39 * (total_words / total_sentences) + 11.8 * (total_syllables / total_words) - 15.59
```

This is a classic readability formula that estimates the U.S. grade level required to understand a text.

In [ ]:
def count_syllables(word):
    """Approximate syllable count for a word."""
    word = word.lower()
    syllables = 0
    vowels = 'aeiouy'
    if word[0] in vowels:
        syllables += 1
    for i in range(1, len(word)):
        if word[i] in vowels and word[i-1] not in vowels:
            syllables += 1
    if word.endswith('e'):
        syllables -= 1
    if word.endswith('le') and len(word) > 2 and word[-3] not in vowels:
        syllables += 1
    return max(1, syllables)

def flesch_kincaid_grade(text):
    """Calculate Flesch-Kincaid Grade Level for a text."""
    import re
    
    # Count sentences (approximate)
    sentences = re.split(r'[.!?]+', text)
    num_sentences = max(1, len([s for s in sentences if s.strip()]))
    
    # Count words
    words = re.findall(r'\b\w+\b', text)
    num_words = max(1, len(words))
    
    # Count syllables
    num_syllables = sum(count_syllables(word) for word in words)
    
    # Calculate FKGL
    asl = num_words / num_sentences  # Average Sentence Length
    asw = num_syllables / num_words  # Average Syllables per Word
    fkgl = 0.39 * asl + 11.8 * asw - 15.59
    
    return max(1, min(12, round(fkgl)))  # Clamp to 1-12

print('Flesch-Kincaid function defined!')

## Apply Readability Scoring

Now we apply our simple readability scorer to all examples and compare with ground-truth labels.

In [ ]:
# Apply Flesch-Kincaid to all examples
results = []
for ex in all_examples:
    predicted_grade = flesch_kincaid_grade(ex['text'])
    results.append({
        'id': ex['id'],
        'text': ex['text'],
        'true_grade': ex['grade_level'],
        'predicted_grade': predicted_grade,
        'error': abs(ex['grade_level'] - predicted_grade)
    })

print(f'Scored {len(results)} examples')
print('\nSample results:')
for r in results[:5]:
    print(f"  {r['id']}: true={r['true_grade']}, predicted={r['predicted_grade']}, error={r['error']}")

## Results and Visualization

Let's visualize the performance of our simple readability scorer.

In [ ]:
# Calculate metrics
true_grades = [r['true_grade'] for r in results]
predicted_grades = [r['predicted_grade'] for r in results]
errors = [r['error'] for r in results]

mae = mean_absolute_error(true_grades, predicted_grades)
accuracy = accuracy_score(true_grades, predicted_grades)

print(f'Mean Absolute Error: {mae:.2f}')
print(f'Exact Match Accuracy: {accuracy:.2%}')
print(f'Average Error: {np.mean(errors):.2f} grade levels')

In [ ]:
# Visualization: True vs Predicted grades
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter plot
axes[0].scatter(true_grades, predicted_grades, alpha=0.6)
axes[0].plot([1, 12], [1, 12], 'r--', label='Perfect prediction')
axes[0].set_xlabel('True Grade Level')
axes[0].set_ylabel('Predicted Grade Level')
axes[0].set_title('True vs Predicted Grade Levels')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error distribution
axes[1].hist(errors, bins=range(0, max(errors)+2), alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Absolute Error (grade levels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Error Distribution')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Print detailed results table
print('=' * 80)
print('DETAILED RESULTS TABLE')
print('=' * 80)
print(f'{"ID":<15} {"True":<6} {"Pred":<6} {"Error":<7} {"Text Preview"}')
print('-' * 80)

for r in results:
    text_preview = r['text'][:40].replace('\n', ' ') + '...'
    print(f"{r['id']:<15} {r['true_grade']:<6} {r['predicted_grade']:<6} {r['error']:<7} {text_preview}")

print('=' * 80)
print(f'Summary: MAE={mae:.2f}, Accuracy={accuracy:.2%}, Avg Error={np.mean(errors):.2f}')
print('=' * 80)